#  **#RAG 챗봇 실습: 「2027 모집요강 Q&A 봇」**


---






> **핵심 질문** : LLM은 *우리 학교 2027학년도 모집요강*을 배운 적이 없다. 그런데 어떻게 정확히, 근거 쪽수까지 붙여 답하게 만들 수 있을까?
> **답** : 모델을 다시 학습시키지 않고, **질문과 관련된 문단을 찾아서(Retrieval) 프롬프트에 붙여(Augmented) 답하게(Generation)** 한다 → **RAG**

| 항목 | 내용 |
|---|---|
| 실습 데이터 | 가천대·건국대 2027학년도 **수시/정시 모집요강 PDF 4개** (총 292쪽) |
| 런타임 | Colab **T4 GPU** 권장 (런타임 > 런타임 유형 변경). CPU만 쓸 때는 설정 셀에서 임베딩 모델을 바꾼다 |
| API 키 | 왼쪽 🔑 **보안 비밀**에 `GEMINI_API_KEY` 등록 → 노트북 액세스 켜기 (없으면 '검색 전용 모드'로 동작) |


가천대학교 입학처 – 수시 모집요강
가천대학교 입학처 – 정시 모집요강
서울과학기술대학교 입학처 – 수시 모집요강
서울과학기술대학교 입학처 – 정시 모집요강
건국대학교 입학처 – 수시 모집요강
건국대학교 입학처 – 정시 모집요강

## 🟠 **RAG 파이프라인** = 색인(미리 한 번) + 질의(질문마다)

| 단계 | 번호 | 상자 | 하는 일 | 이 노트북의 도구 |
|---|---|---|---|---|
| 색인 | ① | 문서 수집 | PDF → 쪽별 텍스트(표는 마크다운) | `pymupdf4llm` |
| 색인 | ② | 청킹 | 검색 단위로 자르기 | 직접 구현(줄 단위 + 겹침) |
| 색인 | ③ | 임베딩 | 문단 → 벡터 | `sentence-transformers` + `BAAI/bge-m3` |
| 색인 | ④ | 벡터 DB | 저장·색인 | `chromadb` |
| 질의 | ⑤ | 질문 | 사용자 입력 | Gradio 입력창 |
| 질의 | ⑥ | 검색 | 상위 k개 청크 | 코사인 유사도 + 메타데이터 필터 |
| 질의 | ⑦ | 프롬프트 조립 | 근거 + 질문 + 규칙 | 직접 구현 |
| 질의 | ⑧ | 생성 | 근거 번호와 함께 답 | Gemini API (`google-genai`) |



## 🟠 **배경 — 검색과 생성은 어떻게 만났나**

```
키워드 검색            의미 검색(밀집 벡터)         검색 + 생성             실무 표준
TF-IDF(1972) ─ BM25(1994) ─ Sentence-BERT(2019) ─ DPR(2020) ─ RAG(2020) ─ 벡터DB·하이브리드·리랭킹(2023~)
"같은 단어가 있나?"     "뜻이 가까운가?"            "찾아서 읽고 답하라"      "근거를 달아 답하라"
```

| 연도 | 사건 | 의미 | 출처 |
|---|---|---|---|
| 1972 | IDF 제안 | 드문 단어일수록 중요하다 (9주차 TF-IDF) | Spärck Jones, *J. of Documentation* 28(1) |
| 1994 | BM25 | 키워드 검색의 표준 | Robertson et al., TREC-3 |
| 2019 | Sentence-BERT | 문장 하나를 벡터 하나로 → 의미 검색이 실용화 | Reimers & Gurevych, EMNLP 2019 (arXiv:1908.10084) |
| 2020 | DPR | 질문·문단을 같은 벡터 공간에 넣어 검색 | Karpukhin et al., EMNLP 2020 (arXiv:2004.04906) |
| 2020 | **RAG** | 검색기 + 생성기를 결합, 'RAG'라는 이름의 시작 | Lewis et al., NeurIPS 2020 (arXiv:2005.11401) |
| 2024 | BGE-M3 | 100개 이상 언어·최대 8,192토큰 임베딩 (이번 실습 모델) | Chen et al. (arXiv:2402.03216) |

---
## 🟠 **예제: 미니 RAG**

진짜 PDF로 가기 전에 **검색 → 프롬프트 조립**의 뼈대만 먼저 본다. (문서는 가상의 동아리 안내문, 검색은 9주차에 배운 TF-IDF)

In [2]:
# [30초 예제] 문서 5줄 + TF-IDF 검색 + 프롬프트 조립 (LLM 호출 직전까지)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [                                   # ① 문서 수집 + ② 청킹 (이미 한 줄씩 잘려 있다고 가정)
    "AI 동아리 '뉴런'의 신입 모집 기간은 3월 2일부터 3월 13일까지이다.",
    "동아리 회비는 학기당 2만 원이며 첫 모임에서 낸다.",
    "정기 모임은 매주 수요일 저녁 6시에 AI관 307호에서 연다.",
    "신입 부원은 파이썬 기초 스터디를 4주 동안 듣는다.",
    "지원서는 동아리 홈페이지의 구글 폼으로 제출한다.",
]
question = "모임은 무슨 요일에 어디서 해?"       # ⑤ 질문

vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 2))   # ③ 임베딩(여기서는 글자 n-gram TF-IDF)
D = vec.fit_transform(docs)                                  # ④ 벡터 DB(여기서는 그냥 행렬)
scores = cosine_similarity(vec.transform([question]), D)[0]  # ⑥ 검색
top = scores.argsort()[::-1][:2]

context = "\n".join(f"[{i+1}] {docs[j]}" for i, j in enumerate(top))
prompt = f"[근거]\n{context}\n\n[질문]\n{question}\n\n[규칙] 근거에 있는 내용만, 근거 번호와 함께 답하라."  # ⑦ 프롬프트 조립
print(prompt)                                                # ⑧ 이 문자열을 LLM에 보내면 RAG 완성

[근거]
[1] 정기 모임은 매주 수요일 저녁 6시에 AI관 307호에서 연다.
[2] 동아리 회비는 학기당 2만 원이며 첫 모임에서 낸다.

[질문]
모임은 무슨 요일에 어디서 해?

[규칙] 근거에 있는 내용만, 근거 번호와 함께 답하라.


**확인** : 질문에는 '요일', 문서에는 '수요일'만 있어도 찾아진다. 그러면 "언제 **만나**?"처럼 **같은 글자가 하나도 없는** 질문은? → 그래서 ③에서 '뜻'을 담는 임베딩 모델이 필요하다.

---
## **① 준비 — 설치와 설정**

In [3]:
# 설치 (Colab 기준 약 1분). sentence-transformers·torch는 Colab에 이미 설치되어 있다.
!pip install -q pymupdf4llm chromadb sentence-transformers "google-genai>=1.55" "gradio>=6"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
t

In [4]:
# ── 실습 설정: 이 셀의 값만 바꿔 가며 실험한다 ─────────────────────────────
DOC_DIR       = "docs"                  # PDF를 넣는 폴더
DB_DIR        = "chroma_db"             # 벡터 DB가 저장되는 폴더
COLLECTION    = "admission_2027"        # 컬렉션(테이블) 이름
UNIVERSITIES  = ["가천대학교", "건국대학교"]  # 문서에서 자동으로 구분할 대학 이름

EMBED_MODEL   = "BAAI/bge-m3"           # GPU 권장. CPU만 있으면 → "intfloat/multilingual-e5-small"
LLM_MODEL     = "gemini-3.6-flash"      # Gemini API 모델 ID (강의 전 공식 문서에서 재확인)

CHUNK_SIZE    = 900                     # 청크 하나의 목표 글자 수
CHUNK_OVERLAP = 150                     # 이웃 청크와 겹치는 글자 수
TOP_K         = 5                       # 질문마다 가져올 청크 수

---
# **Part A. 색인 단계** (미리 한 번)

## ① 문서 수집 — PDF를 '쪽별 텍스트'로

모집요강의 핵심 정보(일정·모집인원·반영비율)는 대부분 **표**에 있다. 표를 그냥 글자로 뽑으면 칸이 한 줄씩 흩어져 "378"이 무엇의 숫자인지 알 수 없게 된다. `pymupdf4llm`은 표를 **마크다운 표**로 바꿔 행·열 관계를 지켜 준다.

| 추출 방식 | 표 한 행의 모습 | RAG에 주는 영향 |
|---|---|---|
| 일반 텍스트 | `일반전형 1` / `가군` / `347` 이 각각 다른 줄 | 숫자와 항목의 연결이 끊어짐 |
| 마크다운 | `\|수능위주\|일반전형 1\|나군\|346\|` | 한 줄에 관계가 보존됨 |

In [5]:
import os, glob, json, re

os.makedirs(DOC_DIR, exist_ok=True)

# PDF가 없으면 업로드 창을 띄운다 (Colab). 4개 파일을 한 번에 선택해서 올린다.
if not glob.glob(f"{DOC_DIR}/*.pdf"):
    try:
        from google.colab import files
        for name, data in files.upload().items():
            with open(os.path.join(DOC_DIR, name), "wb") as f:
                f.write(data)
    except ImportError:
        print(f"로컬 PC에서는 '{DOC_DIR}' 폴더에 PDF를 넣은 뒤 이 셀을 다시 실행하세요.")

pdf_paths = sorted(glob.glob(f"{DOC_DIR}/*.pdf"))
print("PDF 파일:", len(pdf_paths), "개")

PDF 파일: 4 개


In [6]:
import pymupdf4llm

def clean(text):
    # 검색에 필요 없는 마크다운 장식을 지운다 (굵게 표시, 표 칸 안의 줄바꿈 태그 등)
    text = text.replace("**", "").replace("<br>", " ")
    text = re.sub(r"</?mark>", "", text)
    return re.sub(r"[ \t]+", " ", text).strip()

def load_pdf(path):
    # PDF 1개 → [{'page': 쪽 번호, 'text': 그 쪽의 마크다운}, ...]
    pages = pymupdf4llm.to_markdown(path, page_chunks=True, show_progress=False, use_ocr=False)
    result = []
    for i, p in enumerate(pages, start=1):
        page_no = p.get("metadata", {}).get("page_number") or i
        result.append({"page": page_no, "text": clean(p["text"])})
    return result

def detect_label(full_text):
    # 파일명이 아니라 '내용'으로 대학과 전형을 알아낸다 → 파일명이 깨져 있어도 동작
    univ = max(UNIVERSITIES, key=lambda u: full_text.count(u[:3]))        # '가천대' / '건국대' 등장 횟수
    kind = "수시" if full_text.count("수시모집요강") >= full_text.count("정시모집요강") else "정시"
    return univ, kind

# 파싱은 2~3분 걸리므로 결과를 파일로 저장해 두고, 다시 실행할 때는 불러온다(캐시)
CACHE = "pages_cache.json"
if os.path.exists(CACHE):
    documents = json.load(open(CACHE, encoding="utf-8"))
    print("캐시에서 불러옴:", CACHE)
else:
    documents = []
    for path in pdf_paths:
        pages = load_pdf(path)
        univ, kind = detect_label("\n".join(p["text"] for p in pages))
        documents.append({"file": os.path.basename(path), "univ": univ, "kind": kind, "pages": pages})
        print(f"  완료: {univ} {kind} ({len(pages)}쪽)")
    json.dump(documents, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)

import pandas as pd
pd.DataFrame([{"대학": d["univ"], "전형": d["kind"], "쪽수": len(d["pages"]),
               "글자수": sum(len(p["text"]) for p in d["pages"]), "파일": d["file"]} for d in documents])

  완료: 가천대학교 수시 (104쪽)
  완료: 가천대학교 정시 (44쪽)
  완료: 건국대학교 정시 (56쪽)
  완료: 건국대학교 수시 (88쪽)


,대학,전형,쪽수,글자수,파일
0,가천대학교,수시,104,108854,2027가천대학교수시모집요강(업로드).pdf
1,가천대학교,정시,44,37362,2027가천대학교정시모집요강(업로드).pdf
2,건국대학교,정시,56,62155,2027학년도 건국대학교 서울캠퍼스 정시모집요강(단면)_260831.pdf
3,건국대학교,수시,88,106954,2027학년도 건국대학교 수시모집요강(단면)_최종.pdf


In [7]:
# 눈으로 확인: 표가 마크다운으로 잘 뽑혔는가? (가천대 정시 4쪽 = 전형일정)
doc = next(d for d in documents if d["univ"] == "가천대학교" and d["kind"] == "정시")
print(doc["pages"][3]["text"][:700])

2027학년도 정시모집요강 

# 전형일정 

|||구분|일시|비고|
|---|---|---|---|---|
|원서접수|||2027. 01. 04.(월) ~ 07.(목) 18:00||
|서류제출 마감|||2027. 01. 08.(금) 13:00까지|·원서접수 사이트에서 제출|
|실기고사|가군|연기예술학과(연출)|고사장 확인 2027. 01. 11.(월)|· 세부 일정은 개별통지를 하지 않으므로 지원자가 반드시 확인해야 합니다.|
||||전형일 2027. 01. 12.(화)|· 본 대학 입학처 홈페이지에서 고사(실기, 면접) 일정을 반드시 확인해야 합니다.|
||||1단계 발표 2027. 01. 18.(월)|· 전형일은 원서접수 마감 후 지원자 수에 의해 변경될 수 있습니다.|
|면접고사|나군|조기취업형 계약학과 전형|전형일 2027. 01. 21.(목)|· 고사(실기, 면접) 시 본인임을 확인할 수 있는 신분증(주민등록증, 운전면허증, 여권 등)을 지참해야 합니다.|
|합격자 발표||수능위주|2027. 01. 22.(금)||
|||실기, 면접|2027. 02. 04.(목)||
|합격자 등록|||2027. 02. 10.(수) ~ 12.(금)|·본 대학 입학처 홈페이지|
|충원 합격자 발|표 마감||2027. 02. 17.(수) 18:00||
|충원 합격자 등|록 마감||2027. 02. 18.(목) 16:00||



30


**확인** : 대학·전형이 4개 조합(가천 수시/정시, 건국 수시/정시)으로 정확히 잡혔는가? 표의 한 행이 `|...|...|` 한 줄로 보이는가?

## **② 청킹 — 검색 단위로 자르기**

| 질문 | 답 |
|---|---|
| 왜 자르나? | 한 쪽 전체를 벡터 하나로 만들면 여러 주제가 섞여 뜻이 흐려지고, 프롬프트에 넣을 때 토큰도 낭비된다 |
| 너무 작게 자르면? | 표의 머리글·앞뒤 문맥이 잘려 "이 숫자가 무엇인지" 알 수 없다 |
| 겹침(overlap)은 왜? | 경계에 걸친 문장이 양쪽 청크에 모두 남도록 |
| 이 실습의 3가지 장치 | (1) **줄 단위**로 묶어 표의 행을 자르지 않는다 (2) 표 중간에서 잘리면 **표 머리글을 다시 붙인다** (3) 청크 맨 앞에 **`[대학 전형 p.쪽]` 꼬리표**를 붙인다 |

(3)이 특히 중요하다. "원서접수 기간" 표는 네 문서에 모두 있어서, 꼬리표가 없으면 **어느 대학 것인지 벡터에 담기지 않는다.**

In [8]:
def split_page(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # 한 쪽의 텍스트를 '줄 단위'로 묶어 size 글자 안팎의 청크 목록으로 만든다
    lines = [l for l in text.split("\n") if l.strip()]
    chunks, cur, header = [], [], []          # cur: 만들고 있는 청크, header: 지금 지나는 표의 머리글 2줄
    for i, line in enumerate(lines):
        is_row = line.startswith("|")
        if is_row and i + 1 < len(lines) and lines[i + 1].startswith("|---"):
            header = [line, lines[i + 1]]     # 새 표 시작 → 머리글 기억
        elif not is_row:
            header = []                       # 표가 끝남

        if cur and sum(map(len, cur)) + len(line) > size:      # 이 줄을 넣으면 넘친다 → 청크 마감
            chunks.append("\n".join(cur))
            tail, n = [], 0                                    # 겹침: 앞 청크의 끝 줄들을 다시 가져온다
            for prev in reversed(cur):
                if n + len(prev) > overlap:
                    break
                tail.insert(0, prev); n += len(prev)
            in_table = is_row and header and line not in header and header[0] not in tail
            cur = (header if in_table else []) + tail          # 표 중간이면 머리글을 다시 붙인다
        cur.append(line)
    if cur:
        chunks.append("\n".join(cur))
    return chunks

chunks = []
for d in documents:
    for p in d["pages"]:
        for body in split_page(p["text"]):
            if len(body) < 30:                # 표지처럼 내용이 거의 없는 조각은 버린다
                continue
            tag = f"[{d['univ']} 2027학년도 {d['kind']}모집요강 p.{p['page']}]"
            chunks.append({"id": f"c{len(chunks):04d}", "text": f"{tag}\n{body}",
                           "univ": d["univ"], "kind": d["kind"], "page": p["page"], "file": d["file"]})

lens = pd.Series([len(c["text"]) for c in chunks])
print(f"청크 {len(chunks)}개 | 글자 수 중앙값 {int(lens.median())}, 최대 {lens.max()}")
print(pd.DataFrame(chunks).groupby(["univ", "kind"]).size().rename("청크 수"))
print("\n─── 청크 예시 ───\n" + chunks[len(chunks) // 2]["text"][:500])

청크 500개 | 글자 수 중앙값 787, 최대 2875
univ   kind
가천대학교  수시      175
       정시       63
건국대학교  수시      167
       정시       95
Name: 청크 수, dtype: int64

─── 청크 예시 ───
[건국대학교 2027학년도 정시모집요강 p.11]
###### . 전형 요약 및 주요 변경 사항 
###### 1. 전형 요약 
###### ■전형방법 및 모집인원 
||전형명||정원 구분|계열|전형방법 (반영요소 및 반영비율)|모집인원|
|---|---|---|---|---|---|---|
|||가군|내|인문/자연/KU자유전공학부|•�수능 100%|513|
|||||예체능|•�수능 60% + 실기 40%|32|
||KU일반학생|나군|내|인문/자연|•�수능 100%|321|
|||||예체능|•�수능 60% + 실기 40%|146|
|||다군|내|인문/자연|•�수능 100%|249|
|||||예체능|•�수능 60% + 실기 40%|69|
|수능||가군|외|||24|
||기초생활및차상위|나군|외|공통|•수능 100%|35|
|||다군|외|||5|
|||가군|외|||34|
||농어촌학생|나군|외|공통|•�수능 100%|39|
|||다군|외|||5|
||특성화고교졸업자||외|공통|•�수능 


## **③ 임베딩 — 문단을 벡터로**

임베딩 모델은 글을 고정 길이 벡터로 바꾼다. 뜻이 비슷한 글은 벡터의 **방향**이 비슷해지도록 학습되어 있다. 방향이 얼마나 같은지는 **코사인 유사도**로 잰다.

$$\cos(\mathbf{q},\mathbf{d})=\frac{\mathbf{q}\cdot\mathbf{d}}{\lVert\mathbf{q}\rVert\,\lVert\mathbf{d}\rVert}$$

- 분자(내적)는 같은 방향 성분끼리 곱해 더한 값, 분모는 두 벡터의 길이다. 길이로 나누므로 **글의 길이와 무관하게 방향만** 비교한다.
- 값의 범위는 −1~1이고, 1에 가까울수록 뜻이 가깝다.
- 벡터 길이를 미리 1로 맞춰 두면(`normalize_embeddings=True`) 분모가 1이 되어 **내적만 계산하면 된다.** 벡터 DB가 빠른 이유 중 하나다.

| 모델 | 차원 | 최대 입력 | 특징 | 권장 환경 |
|---|---|---|---|---|
| `BAAI/bge-m3` | 1,024 | 8,192토큰 | 다국어, 접두어 불필요 | T4 GPU |
| `intfloat/multilingual-e5-small` | 384 | 512토큰 | 가볍다. 입력 앞에 `query: ` / `passage: ` 접두어 필요 | CPU |

In [9]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=device)
embedder.max_seq_length = 1024              # 이 길이를 넘는 청크는 뒷부분이 잘린다(속도·메모리 절약)
print("임베딩 모델:", EMBED_MODEL, "| 장치:", device)

# E5 계열은 질문과 문단 앞에 서로 다른 접두어를 붙이도록 학습되었다 (bge-m3는 필요 없음)
IS_E5 = "e5" in EMBED_MODEL.lower()

def embed(texts, role="passage"):
    # 문자열 목록 → (개수, 차원) 크기의 넘파이 배열. role은 'passage'(문서) 또는 'query'(질문)
    if IS_E5:
        texts = [f"{role}: {t}" for t in texts]
    return embedder.encode(texts, batch_size=16, normalize_embeddings=True,
                           show_progress_bar=len(texts) > 50)

# 감 잡기: 글자는 달라도 뜻이 가까우면 유사도가 높다
demo_vecs = embed(["원서접수 기간이 언제인가요?", "지원서는 며칠까지 내야 하나요?", "기숙사 식당 메뉴가 궁금해요"], "query")
print("벡터 모양:", demo_vecs.shape)
print("원서접수 ↔ 지원서 제출 :", round(float(demo_vecs[0] @ demo_vecs[1]), 3))
print("원서접수 ↔ 기숙사 식당 :", round(float(demo_vecs[0] @ demo_vecs[2]), 3))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

임베딩 모델: BAAI/bge-m3 | 장치: cuda
벡터 모양: (3, 1024)
원서접수 ↔ 지원서 제출 : 0.823
원서접수 ↔ 기숙사 식당 : 0.486


## **④ 벡터 DB — 저장·색인**

벡터 DB는 (1) 벡터 (2) 원문 (3) 메타데이터를 함께 저장하고, 질문 벡터와 가까운 것을 빨리 찾아 준다. Chroma는 내부적으로 **HNSW**(근사 최근접 이웃 그래프) 색인을 쓴다. 청크가 수백 개인 지금은 전수 비교도 충분하지만, 수백만 개가 되면 색인이 필수다.

| 저장하는 것 | 예 | 쓰임 |
|---|---|---|
| id | `c0123` | 중복 방지·갱신 |
| embedding | 1,024차원 벡터 | ⑥ 검색 |
| document | 청크 원문 | ⑦ 프롬프트의 근거 |
| metadata | 대학·전형·쪽 | 필터링, 출처 표시 |

In [10]:
import chromadb

client = chromadb.PersistentClient(path=DB_DIR)       # 폴더에 저장 → 런타임이 살아 있는 동안 재사용
try:
    client.delete_collection(COLLECTION)              # 이 셀을 다시 실행해도 중복되지 않도록 비운다
except Exception:
    pass
col = client.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})   # 거리 = 1 - 코사인 유사도

vectors = embed([c["text"] for c in chunks])          # ③ 전체 청크 임베딩 (T4에서 1분 안팎)

for i in range(0, len(chunks), 500):                  # 500개씩 나눠 넣는다
    batch = chunks[i:i + 500]
    col.add(ids=[c["id"] for c in batch],
            embeddings=vectors[i:i + 500].tolist(),
            documents=[c["text"] for c in batch],
            metadatas=[{"univ": c["univ"], "kind": c["kind"], "page": c["page"], "file": c["file"]} for c in batch])

print("벡터 DB에 저장된 청크:", col.count(), "개 | 벡터 차원:", vectors.shape[1])

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

벡터 DB에 저장된 청크: 500 개 | 벡터 차원: 1024


---
# **Part B. 질의 단계** (질문마다)


## **⑤ 질문 → ⑥ 검색 — 상위 k개 청크**

질문도 **같은 임베딩 모델**로 벡터로 바꾼 뒤, DB에서 가장 가까운 k개를 꺼낸다. 메타데이터 필터(`where`)를 쓰면 "가천대 정시만"처럼 범위를 좁힐 수 있다.

In [11]:
def retrieve(question, k=TOP_K, univ="전체", kind="전체"):
    # 질문과 가장 가까운 청크 k개를 [{text, univ, kind, page, score}, ...] 로 돌려준다
    conds = []
    if univ != "전체": conds.append({"univ": univ})
    if kind != "전체": conds.append({"kind": kind})
    where = None if not conds else conds[0] if len(conds) == 1 else {"$and": conds}

    res = col.query(query_embeddings=embed([question], "query").tolist(), n_results=int(k), where=where)
    return [{"text": text, **meta, "score": round(1 - dist, 3)}          # 코사인 거리 → 유사도
            for text, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

def show_hits(hits):
    return pd.DataFrame([{"번호": i, "대학": h["univ"], "전형": h["kind"], "쪽": h["page"], "유사도": h["score"],
                          "미리보기": h["text"].split("\n", 1)[-1][:60].replace("\n", " ")} for i, h in enumerate(hits, 1)])

question = "가천대학교 정시 원서접수 기간은 언제야?"
hits = retrieve(question)
show_hits(hits)

,번호,대학,전형,쪽,유사도,미리보기
0,1,가천대학교,정시,8,0.753,# 원서접수 |원서접수 기간|모집시기|접수기간 인터넷 접수| |---|---|--...
1,2,가천대학교,정시,4,0.744,2027학년도 정시모집요강 # 전형일정 |||구분|일시|비고| |---|---|...
2,3,가천대학교,정시,32,0.738,2027학년도 정시모집요강 # 제출서류 안내사항 ###### 서류 제출 방법 ...
3,4,가천대학교,수시,35,0.738,2027학년도 수시모집요강 ###### 원서접수 |원서접수 기간|모집시기|접수기...
4,5,가천대학교,수시,83,0.727,2027학년도 수시모집요강 ###### 제출서류 안내사항 ###### 서류 제출...


**확인** : 상위 결과에 *가천대 정시 4쪽(전형일정)* 이 들어 있는가? `retrieve(question, univ="가천대학교", kind="정시")`처럼 필터를 걸면 결과가 어떻게 달라지는가?

## **⑦ 프롬프트 조립 — 근거 + 질문 + 규칙**

검색된 청크에 **번호**를 붙여 넣고, LLM이 지켜야 할 **규칙**을 준다. 규칙은 대화 내내 유지되어야 하므로 시스템 지시문(`system_instruction`)으로 보낸다.

| 구성 요소 | 내용 | 없으면 생기는 문제 |
|---|---|---|
| 근거 | `[1] (대학 전형 p.쪽)` + 청크 원문 | LLM이 기억에 의존해 그럴듯한 오답을 만든다(환각) |
| 질문 | 사용자 입력 그대로 | — |
| 규칙 | 근거만 사용 · 번호 표시 · 없으면 모른다고 답하기 | 출처를 확인할 수 없고, 근거 밖의 내용이 섞인다 |

In [12]:
SYSTEM_RULES = '''너는 대학 입학 모집요강 안내 도우미다. 다음 규칙을 반드시 지켜라.
1. [근거]에 적힌 내용만 사용해 한국어로 답한다. 네가 원래 알고 있던 지식은 쓰지 않는다.
2. 근거를 사용한 문장 끝에는 [1], [2]처럼 근거 번호를 붙인다.
3. 근거에 답이 없으면 "제공된 모집요강에서 찾을 수 없습니다."라고 답하고 추측하지 않는다.
4. 대학이나 전형(수시/정시)이 다른 근거를 섞지 않는다. 질문이 대학을 지정하지 않았으면 대학별로 나눠 답한다.
5. 날짜·인원·비율 같은 수치는 근거에 적힌 표기 그대로 옮긴다.
6. 답은 간결하게 쓰고, 마지막 줄에 "※ 최종 확인은 각 대학 입학처 공지에서 하세요."를 붙인다.'''

def build_prompt(question, hits):
    blocks = []
    for i, h in enumerate(hits, 1):
        body = h["text"].split("\n", 1)[-1]                       # 꼬리표를 떼고 본문만
        blocks.append(f"[{i}] ({h['univ']} {h['kind']}모집요강 p.{h['page']})\n{body}")
    return "[근거]\n" + "\n\n".join(blocks) + f"\n\n[질문]\n{question}"

prompt = build_prompt(question, hits)
print(prompt[:900], "\n...\n(프롬프트 전체 길이:", len(prompt), "글자)")

[근거]
[1] (가천대학교 정시모집요강 p.8)
# 원서접수 
|원서접수 기간|모집시기|접수기간 인터넷 접수|
|---|---|---|
||정시 2027.|01. 04.(월) ~ 07.(목) 본 대학 입학처 홈페이지|
|원서접수 방법|인터넷 원서접수 시 사진 업로드 [최근 3개월 내 사|를 위하여 지원자 본인의 증명사진 파일(jpg, gif 파일형식)을 준비하시기 바랍니다. 진으로 인물 뒤 배경이 있는 사진 또는 스냅사진은 사용이 불가함]|
||원서접수 사이트 접속|가천대학교 입학처 홈페이지 접속 → 입학원서 접수 대행기관|
||||
||회원가입 및 로그인|지원자 본인 명의로 회원가입|
||||
||유의사항 확인|유의사항을 반드시 확인하여야 하며, 미확인으로 인한 책임은 지원자에게 있음|
||||
||원서작성|① 모집요강을 참고하여 전형유형, 지원학과 / 전공 등을 선택하여 입력 ② 모든 사항을 빠짐없이 정확하게 입력 및 확인(학생부 온라인 제공 동의)|
||||
||전형료 결제|전형료 결제 후에는 입학원서 기재사항을 수정하거나 원서접수를 취소할 수 없으며, 전형료는 반환하지 않음|
||||
||수험표 확인|접수가 완료된 것을 원서와 수험표를 통해 직접 확인|
||||
||서류제출(해당자만)|① 온라인 원서접수 사이트를 통해 제출 ② 각각의 제출서류를 저용량 PDF로 합본하여 한 개의 문서로 제출해야 합니다.|
1. 원서 및 서류제출은 온라인으로 접수합니다. 
2. 본 대학에 원서를 접수하면 해당 전형과 관련된 학교생활기록부 및 수능성적 자료 온라인 제공에 동의하는 것 으로 간주합니다. 
3. 장애인복지법 제32조에 의하여 장애인등록을 필하고, 각종 장애 또는 지체로 인하여 입학전형 진행과정에서 지원이 필요한 경우 사전 요청바랍니다. 
4. 장애학생의 지원 및 선발에 대한 
...
(프롬프트 전체 길이: 4124 글자)


## **⑧ 생성 — 근거 번호와 함께 답**

조립한 프롬프트를 LLM에 보낸다. 여기서는 Gemini API의 **Interactions API**를 쓴다(Google 공식 문서 기준 2026-06부터 기본 인터페이스).

> ★ 참고  API 키가 없어도 실습이 멈추지 않도록 **검색 전용 모드**를 넣었다. 이때는 ⑥ 검색 결과까지만 확인하고, ⑧은 키를 받은 뒤 실행한다.

In [43]:
import time
from google.colab import userdata
from google import genai
from google.genai import types

# 코랩 보안 비밀에서 API 키 읽기
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("코랩 보안 비밀에 GEMINI_API_KEY를 설정하세요.")

# timeout은 밀리초 단위: 30,000ms = 30초
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=30_000),
)

print("① API 호출 직전", flush=True)
start = time.monotonic()

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",  # gemini-3.8-flash
        contents="안녕하세요. 한 문장으로 인사해 주세요.",
        config=types.GenerateContentConfig(max_output_tokens=100),
    )

    print(
        "② API 응답 수신:",
        round(time.monotonic() - start, 1),
        "초",
        flush=True,
    )
    print("③ 응답 내용:", response.text, flush=True)

except Exception as e:
    print(
        "② 오류 발생까지:",
        round(time.monotonic() - start, 1),
        "초",
        flush=True,
    )
    print("③ 오류 종류:", type(e).__name__, flush=True)
    print("④ 오류 내용:", str(e)[:1000], flush=True)

finally:
    client.close()

① API 호출 직전
② API 응답 수신: 1.1 초
③ 응답 내용: 안녕하세요! 오늘 어떤 도움이 필요하신가요?


In [45]:
import os
from google import genai

# OpenAI 및 Anthropic 라이브러리가 설치되어 있지 않을 경우를 대비한 자동 설치 및 임포트
try:
    import openai
except ImportError:
    !pip install -q openai
    import openai

try:
    import anthropic
except ImportError:
    !pip install -q anthropic
    import anthropic

def get_api_key(name):
    try:                                   # Colab: 왼쪽 🔑 보안 비밀
        from google.colab import userdata
        val = userdata.get(name)
        return val.strip() if val else None
    except Exception:                      # 로컬 PC: 환경 변수
        val = os.environ.get(name)
        return val.strip() if val else None

# API 키 수집
GEMINI_KEY = get_api_key("GEMINI_API_KEY")
OPENAI_KEY = get_api_key("OPENAI_API_KEY")

# 원래의 ANTHROPIC_API_KEY 환경 변수 불순물(\r\n) 정제 및 추출
ANTHROPIC_KEY = get_api_key("ANTHROPIC_API_KEY")

# 클라이언트 풀 구성
clients = {}
if GEMINI_KEY:
    # 503 회피를 위해 1차 타겟은 gemini-1.5-flash로 설정 (유연한 fallback 내장)
    clients["Gemini"] = {
        "client": genai.Client(api_key=GEMINI_KEY),
        "models": ["gemini-3.5-flash-lite", "gemini-3.8-flash"]
    }
if OPENAI_KEY:
    clients["OpenAI"] = {
        "client": openai.OpenAI(api_key=OPENAI_KEY),
        "models": ["gpt-4o-mini", "gpt-3.5-turbo"]
    }
if ANTHROPIC_KEY:
    clients["Anthropic"] = {
        "client": anthropic.Anthropic(api_key=ANTHROPIC_KEY),
        "models": ["claude-haiku-5-5"]
    }

# 기본 제공자 설정
llm_provider = None
if GEMINI_KEY:
    llm_provider = "Gemini"
elif ANTHROPIC_KEY:
    llm_provider = "Anthropic"
elif OPENAI_KEY:
    llm_provider = "OpenAI"


active_models = clients[llm_provider]["models"] if llm_provider else []
print("LLM 활성화 정보:", f"{llm_provider} (우선순위: {active_models})" if llm_provider else "키 없음 → 검색 전용 모드")

def generate(prompt, system=SYSTEM_RULES):
    if not clients:
        return "⚠️ 등록된 API 키가 없어 **검색 전용 모드**입니다. 위의 '검색된 근거'를 직접 읽고 답을 찾아보세요."

    # 사용 가능한 API 리스트 순회하며 유연하게 대체 호출 (폴백 구현)
    providers_to_try = [llm_provider] + [p for p in clients if p != llm_provider]
    errors = []

    for provider in providers_to_try:
        if provider not in clients:
            continue

        cfg = clients[provider]
        client = cfg["client"]
        models = cfg["models"]

        # 현재 제공자의 지원 가능 모델 목록 순회
        for model in models:
            try:
                if provider == "Gemini":
                    res = client.models.generate_content(
                        model=model,
                        contents=prompt,
                        config={"system_instruction": system}
                    )
                    return res.text

                elif provider == "OpenAI":
                    res = client.chat.completions.create(
                        model=model,
                        messages=[
                            {"role": "system", "content": system},
                            {"role": "user", "content": prompt}
                        ]
                    )
                    return res.choices[0].message.content

                elif provider == "Anthropic":
                    res = client.messages.create(
                        model=model,
                        max_tokens=2048,
                        system=system,
                        messages=[
                            {"role": "user", "content": prompt}
                        ]
                    )
                    return res.content[0].text
            except Exception as e:
                errors.append(f"{provider}({model}): {e}")
                continue # 현재 제공자의 다음 모델 혹은 다음 제공자 시도

    return f"⚠️ 모든 LLM 호출 실패: {', '.join(errors)}"

def rag_answer(question, k=TOP_K, univ="전체", kind="전체"):
    hits = retrieve(question, k, univ, kind)               # ⑥ 검색
    answer = generate(build_prompt(question, hits))        # ⑦ 조립 → ⑧ 생성
    return answer, hits

answer, hits = rag_answer("가천대학교 정시 원서접수 기간은 언제야?")
print(answer)

LLM 활성화 정보: Gemini (우선순위: ['gemini-3.5-flash-lite', 'gemini-3.8-flash'])
가천대학교 정시 원서접수 기간은 2027. 01. 04.(월) ~ 07.(목)이다 [1], [2]. (단, 원서접수 마감일인 2027. 01. 07.(목)은 18:00까지이다 [2].)

※ 최종 확인은 각 대학 입학처 공지에서 하세요.


In [46]:
# RAG가 없으면? 같은 질문을 근거 없이 보내 본다 → 두 답을 모집요강 원문과 대조
q = "건국대학교 2027학년도 정시 원서접수는 며칠 몇 시에 마감해?"
print("■ RAG 없음\n", generate(q, system="한국어로 간결하게 답하라."))
print("\n■ RAG 있음\n", rag_answer(q)[0])

■ RAG 없음
 2027학년도 건국대학교 정시 원서접수 마감 일정은 아직 공식 발표되지 않았습니다. 보통 1차 정시 원서접수는 1월 초에 진행됩니다. 대학 입학처 홈페이지를 확인해 주세요.

■ RAG 있음
 건국대학교 2027학년도 정시모집 원서접수는 2027년 1월 7일(목) 17:00에 마감합니다 [1], [2], [4].

※ 최종 확인은 각 대학 입학처 공지에서 하세요.


**확인** : 'RAG 없음' 답의 날짜·시각은 믿을 수 있는가? 'RAG 있음' 답의 `[번호]`를 따라가 원문과 대조해 보자. **근거 번호가 있어야 검증할 수 있다.**

---
# **Part C. Gradio 챗봇으로 묶기**

`gr.ChatInterface`는 `fn(message, history, *추가입력)` 함수 하나만 주면 챗봇 화면을 만들어 준다. 여기서는 **검색된 근거를 먼저** 접이식 상자로 보여 주고, 이어서 답을 보여 준다(파이프라인 ⑥ → ⑧ 순서가 화면에 그대로 드러난다).

| UI 요소 | 파이프라인 | 코드 |
|---|---|---|
| 입력창 | ⑤ 질문 | `message` |
| 대학·전형 선택, k 슬라이더 | ⑥ 검색 조건 | `additional_inputs` |
| 🔎 검색된 근거(접이식) | ⑥ 검색 결과 | `gr.ChatMessage(metadata={"title": ...})` |
| 답변 | ⑧ 생성 | `gr.ChatMessage(content=...)` |

In [47]:
import gradio as gr

def format_sources(hits):
    # 검색된 청크를 사람이 읽기 좋은 마크다운으로
    lines = []
    for i, h in enumerate(hits, 1):
        preview = h["text"].split("\n", 1)[-1][:150].replace("\n", " ").replace("|", " ")
        lines.append(f"**[{i}] {h['univ']} {h['kind']} p.{h['page']}** · 유사도 {h['score']}\n\n> {preview}…")
    return "\n\n".join(lines) if lines else "조건에 맞는 근거를 찾지 못했습니다."

def respond(message, history, univ, kind, k):
    hits = retrieve(message, k, univ, kind)                               # ⑥ 검색
    sources = gr.ChatMessage(role="assistant", content=format_sources(hits),
                             metadata={"title": f"🔎 검색된 근거 {len(hits)}개 (눌러서 펼치기)"})
    yield [sources]                                                       # 근거를 먼저 화면에 표시
    answer = generate(build_prompt(message, hits)) if hits else "제공된 모집요강에서 찾을 수 없습니다."
    yield [sources, gr.ChatMessage(role="assistant", content=answer)]     # ⑦ 조립 → ⑧ 생성

demo = gr.ChatInterface(
    fn=respond,
    title="🎓 2027 모집요강 Q&A 봇 (RAG)",
    description="가천대·건국대 2027학년도 수시/정시 모집요강 PDF에서 근거를 찾아 답합니다. 답변의 [번호]는 '검색된 근거'의 번호입니다.",
    additional_inputs=[
        gr.Dropdown(["전체"] + UNIVERSITIES, value="전체", label="대학"),
        gr.Radio(["전체", "수시", "정시"], value="전체", label="전형"),
        gr.Slider(1, 10, value=TOP_K, step=1, label="검색할 청크 수 (k)"),
    ],
    additional_inputs_accordion=gr.Accordion("검색 조건", open=True),
    examples=[
        ["가천대학교 정시 원서접수 기간은 언제야?", "전체", "전체", TOP_K],
        ["건국대 수시 원서접수 기간과 접수 사이트를 알려줘", "건국대학교", "수시", TOP_K],
        ["정시 일반전형1은 군별로 몇 명을 뽑아?", "가천대학교", "정시", TOP_K],
        ["두 대학의 정시 원서접수 마감 일시를 비교해줘", "전체", "정시", 8],
    ],
)
demo.launch(share=True, debug=True)   # Colab: 공개 주소(gradio.live)가 출력된다. 멈추려면 셀 정지(■)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d360ed0d06688b835f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d360ed0d06688b835f.gradio.live


---
**# Part D. 검색 품질 재 보기 — Hit@k**

RAG의 답이 틀렸다면 원인은 둘 중 하나다: **검색이 못 찾았거나(⑥)**, **찾았는데 LLM이 잘못 읽었거나(⑧)**. 먼저 검색부터 숫자로 확인한다.

- **Hit@k** : 정답이 들어 있는 청크가 상위 k개 안에 있으면 1, 없으면 0. 여러 질문의 평균이 Hit@k이다.
- 아래 '정답 문자열'은 모집요강 원문에서 직접 확인한 표기다.

> 위 Gradio 셀이 실행 중이면 먼저 정지(■)한 뒤 실행한다.

In [49]:
TESTSET = [   # (질문, 대학, 전형, 근거 청크에 들어 있어야 하는 정답 문자열)
    ("가천대학교 정시 원서접수 기간은?",        "가천대학교", "정시", "2027. 01. 04"),
    ("건국대학교 정시 원서접수 기간은?",        "건국대학교", "정시", "2027. 1. 5"),
    ("가천대학교 수시 원서접수 기간은?",        "가천대학교", "수시", "2026. 09. 07"),
    ("건국대학교 수시 원서접수 기간은?",        "건국대학교", "수시", "2026. 9. 8"),
    ("가천대 정시 일반전형1 다군 모집인원은?",  "가천대학교", "정시", "378"),
]

def norm(s):
    # 공백과 표 구분선(|)을 지우고 비교한다 → '2027.|01. 04'처럼 칸이 어긋난 추출 결과도 정답으로 인정
    return re.sub(r"[\s|]", "", s)

def hit_at_k(k=TOP_K, use_filter=False):
    rows = []
    for q, univ, kind, gold in TESTSET:
        hits = retrieve(q, k, univ, kind) if use_filter else retrieve(q, k)
        ok = any(norm(gold) in norm(h["text"]) and h["univ"] == univ and h["kind"] == kind for h in hits)
        rows.append({"질문": q, "정답 문자열": gold, "Hit": int(ok)})
    return pd.DataFrame(rows)

result = hit_at_k(k=TOP_K)
print(f"Hit@{TOP_K} (필터 없음) = {result['Hit'].mean():.2f}")
print(f"Hit@{TOP_K} (대학·전형 필터) = {hit_at_k(TOP_K, use_filter=True)['Hit'].mean():.2f}")
result

Hit@5 (필터 없음) = 1.00
Hit@5 (대학·전형 필터) = 1.00


,질문,정답 문자열,Hit
0,가천대학교 정시 원서접수 기간은?,2027. 01. 04,1
1,건국대학교 정시 원서접수 기간은?,2027. 1. 5,1
2,가천대학교 수시 원서접수 기간은?,2026. 09. 07,1
3,건국대학교 수시 원서접수 기간은?,2026. 9. 8,1
4,가천대 정시 일반전형1 다군 모집인원은?,378,1


---
# 도전 실습 과제 3개

| 번호 | 과제 | 해야 할 일 | 확인할 것 |
|---|---|---|---|
| 1 |**청크 크기 실험** | 설정 셀의 `CHUNK_SIZE`를 300 / 900 / 2000으로 바꿔 ②~④를 다시 실행하고 Part D의 Hit@k를 표로 정리 | 작을수록 좋은가, 클수록 좋은가? 왜 그런가? 꼬리표(`[대학 전형 p.쪽]`)를 빼면 Hit@k가 어떻게 변하는가? |
| 2 | **하이브리드 검색** | `rank_bm25`로 키워드 검색을 추가하고, 벡터 검색 순위와 **RRF**(Reciprocal Rank Fusion: 점수 = Σ 1/(60+순위))로 합치기 | "378명", "논술(일반)"처럼 **숫자·고유 명칭**이 들어간 질문에서 순위가 어떻게 바뀌는가? |
| 3 |  **내 문서 RAG 봇 + 대화 맥락** | (a) 우리 학과 교육과정·학칙 등 자신의 PDF로 교체 (b) "그럼 건국대는?" 같은 **후속 질문**이 검색되도록, `history`를 이용해 LLM으로 질문을 독립된 문장으로 다시 쓴 뒤 검색 (c) 직접 만든 질문 10개로 Hit@k와 답변 정확도를 보고 | 질문 다시 쓰기 전후의 Hit@k 차이. 근거에 없는 질문에 "찾을 수 없습니다"라고 답하는가? |

## 생각해 볼 질문

| 질문 | 힌트 |
|---|---|
| 표가 두 쪽에 걸쳐 있으면? | 이 실습은 쪽 단위로 자르므로 둘째 쪽에는 머리글이 없다 → 쪽 경계를 넘는 청킹이 필요 |
| 추출 오류는 없는가? | 가천대 수시 34쪽 일정표에 `18:0\|0`처럼 칸이 어긋난 곳이 있다. **조용히 틀리는 데이터**는 RAG가 고쳐 주지 않는다 |
| 유사도 0.6은 높은가? | 절대값은 모델마다 다르다. **순위**와 **1·2위의 차이**를 본다 |
| 파인튜닝 대신 RAG를 쓰는 이유는? | 문서가 바뀌면 색인만 다시 하면 되고, 근거를 보여 줄 수 있다 (10주차 LoRA와 비교) |

## 참고 문헌·공식 문서

| 구분 | 출처 |
|---|---|
| RAG | Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", NeurIPS 2020, arXiv:2005.11401 |
| 밀집 검색 | Karpukhin et al., "Dense Passage Retrieval for Open-Domain Question Answering", EMNLP 2020, arXiv:2004.04906 |
| 문장 임베딩 | Reimers & Gurevych, "Sentence-BERT", EMNLP 2019, arXiv:1908.10084 |
| 임베딩 모델 | Chen et al., "BGE M3-Embedding", 2024, arXiv:2402.03216 |
| HNSW | Malkov & Yashunin, IEEE TPAMI 42(4), 2020, arXiv:1603.09320 |
| RRF | Cormack, Clarke & Büttcher, "Reciprocal Rank Fusion…", SIGIR 2009 |
| Gemini API | Google AI for Developers, Interactions API 문서 — https://ai.google.dev/gemini-api/docs/interactions-overview |
| Gradio | ChatInterface 문서 — https://www.gradio.app/docs/gradio/chatinterface |
| Chroma | 공식 문서 — https://docs.trychroma.com |
| 실습 데이터 | 가천대학교·건국대학교 입학처, 2027학년도 수시/정시 모집요강 (2026 발행) |